# Stg Layer — channels (Incremental)

Read **channels** từ Bronze (`prod.bronze.channels_raw`, append-only copy of Lakebase Postgres) (Databricks App ghi vào khi user search/save video) → MERGE 1:1 vào Delta staging table. Không transform, không join — đúng schema nguồn + 2 cột ETL metadata.

**Incremental Flow:**
1. Lấy watermark (`MAX(COALESCE(updated_at, created_at))`) từ Delta table
2. SELECT từ `prod.bronze.channels_raw` (Bronze): chỉ rows có `COALESCE(updated_at, created_at) > watermark`
3. MERGE vào `prod.silver.stg_channels` (UPSERT on `channel_id` + `snapshot_date`)
4. Verify row count

**Kiến trúc:** Databricks App → user search/save YouTube data → INSERT/UPDATE vào Postgres → `Analytic/bronze/ingest_raw` (append vào Bronze) → Job notebook (this) → SELECT Bronze (incremental) → MERGE vào Delta (stg layer, 1:1 với nguồn).


**Grain:** 1 row per channel per `snapshot_date` (the Vietnam-time day the app observed the counters). A rerun on the same day updates that day's row; earlier days are kept, so the counters' history is preserved. A row only appears on days the app refreshed this channel (its `updated_at` moved), so days without a refresh have no row.

> **Source change (sub-task 3):** reads `prod.bronze.channels_raw` instead of `lakebase_pg` and writes `prod.silver.stg_channels`. Bronze keeps every read, so rows are deduped to the latest read per grain key before the MERGE.

In [0]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

SOURCE_TABLE = f"{BRONZE}.channels_raw"  # Bronze: append-only copy of every Postgres read
DELTA_TABLE = f"{SILVER}.stg_channels"
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"  # business day used for snapshot_date
MERGE_KEY = "channel_id"

print(f"Bronze source: {SOURCE_TABLE}")
print(f"Delta target: {DELTA_TABLE}")

In [0]:
# --- Get watermark from Delta table (last successful ETL) ---
try:
    watermark_row = spark.sql(f"SELECT MAX(COALESCE(updated_at, created_at)) AS wm FROM {DELTA_TABLE}").collect()[0]
    watermark = watermark_row["wm"] if watermark_row else None
except Exception:
    watermark = None

if watermark:
    # Incremental: only rows changed after watermark
    # Use Spark SQL — timestamps stay in Spark's timezone (UTC), no JDBC mismatch
    src_df = spark.sql(f"""
        SELECT * FROM {SOURCE_TABLE}
        WHERE COALESCE(updated_at, created_at) > '{watermark.strftime("%Y-%m-%d %H:%M:%S.%f")}'
    """)
    print(f"💧 Watermark: {watermark}")
    print("Incremental load: rows after watermark")
else:
    # Full load (first run or Delta empty)
    src_df = spark.sql(f"SELECT * FROM {SOURCE_TABLE}")
    print("📦 Full load (no watermark — first run)")

src_count = src_df.count()
print(f"Read {src_count} rows from Bronze ✅")
if src_count > 0:
    display(src_df.limit(10))

In [0]:
# --- Add snapshot_date, etl_date and op_type to source data ---
from pyspark.sql.functions import current_timestamp, lit, to_date, from_utc_timestamp, coalesce, col
# snapshot_date = the day (Vietnam time) the app observed these counters
src_df = src_df.withColumn(
    "snapshot_date",
    to_date(from_utc_timestamp(coalesce(col("updated_at"), col("created_at")), SNAPSHOT_TZ)),
)
# --- Keep the latest Bronze read per grain key (Bronze holds every read) ---
from pyspark.sql import Window
from pyspark.sql.functions import row_number
latest_read = Window.partitionBy("channel_id", "snapshot_date").orderBy(col("ingest_ts").desc())
src_df = (
    src_df.withColumn("_rn", row_number().over(latest_read))
    .filter("_rn = 1")
    .drop("_rn", "ingest_ts", "ingest_date")
)
src_df = src_df.withColumn("etl_date", current_timestamp())
src_df = src_df.withColumn("op_type", lit("INSERT"))

# --- Create Delta table if not exists (1:1 with source + ETL metadata) ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DELTA_TABLE} (
        channel_id        STRING,
        snapshot_date     DATE,
        channel_title     STRING,
        channel_description STRING,
        custom_url        STRING,
        country           STRING,
        default_language  STRING,
        published_at      TIMESTAMP,
        subscriber_count  BIGINT,
        view_count        BIGINT,
        video_count       BIGINT,
        thumbnail_url     STRING,
        created_at        TIMESTAMP,
        updated_at        TIMESTAMP,
        source            STRING,
        etl_date       TIMESTAMP,
        op_type        STRING
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"Delta table {DELTA_TABLE} ready ✅")

# --- Migrate a table created before snapshot_date existed (runs once) ---
if "snapshot_date" not in [f.name for f in spark.table(DELTA_TABLE).schema.fields]:
    spark.sql(f"ALTER TABLE {DELTA_TABLE} ADD COLUMNS (snapshot_date DATE)")
    spark.sql(f"""
        UPDATE {DELTA_TABLE}
        SET snapshot_date = to_date(from_utc_timestamp(COALESCE(updated_at, created_at), '{SNAPSHOT_TZ}'))
        WHERE snapshot_date IS NULL
    """)
    print(f"Added snapshot_date to {DELTA_TABLE} and backfilled existing rows ✅")

# --- Migrate: source (task that wrote the Postgres row last, added 2026-10-31; older rows NULL) ---
if "source" not in [f.name for f in spark.table(DELTA_TABLE).schema.fields]:
    spark.sql(f"ALTER TABLE {DELTA_TABLE} ADD COLUMNS (source STRING)")
    print(f"Added source to {DELTA_TABLE} ✅")
if "source" not in src_df.columns:  # Bronze rows read before Postgres had the column
    src_df = src_df.withColumn("source", lit(None).cast("string"))

# --- MERGE via SQL (compact, op_type tracking) ---
src_df.createOrReplaceTempView("src_source")

spark.sql(f"""
    MERGE INTO {DELTA_TABLE} AS t
    USING src_source AS s ON t.channel_id = s.channel_id AND t.snapshot_date = s.snapshot_date
    WHEN MATCHED THEN UPDATE SET
        channel_title=s.channel_title,
        channel_description=s.channel_description,
        custom_url=s.custom_url,
        country=s.country,
        default_language=s.default_language,
        published_at=s.published_at,
        subscriber_count=s.subscriber_count,
        view_count=s.view_count,
        video_count=s.video_count,
        thumbnail_url=s.thumbnail_url,
        updated_at=s.updated_at,
        source=s.source,
        etl_date=s.etl_date, op_type='UPDATE'
    WHEN NOT MATCHED THEN INSERT *
""")

merged_count = spark.sql(f"SELECT COUNT(*) FROM {DELTA_TABLE}").collect()[0][0]
print(f"✅ Merged into {DELTA_TABLE}: {merged_count} total rows")

In [0]:
# --- Verify ---
total_delta = spark.sql(f"SELECT COUNT(*) as cnt FROM {DELTA_TABLE}").collect()[0]["cnt"]
last_etl = spark.sql(f"SELECT MAX(etl_date) as etl FROM {DELTA_TABLE}").collect()[0]["etl"]

op_counts = spark.sql(f"SELECT op_type, COUNT(*) as cnt FROM {DELTA_TABLE} GROUP BY op_type").collect()
op_breakdown = ", ".join(f"{r['op_type']}: {r['cnt']}" for r in op_counts)

print(f"📊 Total rows in {DELTA_TABLE}: {total_delta}")
print(f"📊 Last ETL date: {last_etl}")
print(f"📊 Op type breakdown: {op_breakdown}")
print(f"📊 Rows loaded this run: {src_count}")
grain = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT channel_id, snapshot_date) AS n_grain, COUNT(DISTINCT snapshot_date) AS n_days FROM {DELTA_TABLE}").collect()[0]
print(f"📊 Grain check (channel_id × snapshot_date): {grain['n']} rows vs {grain['n_grain']} distinct, over {grain['n_days']} day(s) " + ("✅" if grain["n"] == grain["n_grain"] else "❌ duplicates"))
print("\n--- stg_channels ETL complete ---")